In [1]:
import pandas as pd
import numpy as np
import os

# ── CONFIGURATION ────────────────────────────────────────────
# If run from notebooks directory, change working directory to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

INPUT_PATH = os.path.join("data", "raw", "OnlineRetail_raw.csv")
OUTPUT_PATH = os.path.join("data", "cleaned", "OnlineRetail_cleaned.csv")

print("=" * 60)
print("NOTEBOOK 02: DATA CLEANING & PREPROCESSING")
print("=" * 60)

try:
    print(f"\n[INFO] Loading raw backup from: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    original_rows = len(df)
    print(f"[OK]   Loaded {original_rows:,} rows successfully.\n")
    
    current_rows = original_rows
    
    # ── STEP 1: Remove Cancellations ─────────────────────────
    # Business Rationale: Cancellations distort customer retention metrics
    # and revenue analysis, as they represent failed transactions.
    print("Step 1: Removing Cancellations...")
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    cancellation_mask = df['InvoiceNo'].str.startswith('C')
    cancellation_count = cancellation_mask.sum()
    df = df[~cancellation_mask]
    new_rows = len(df)
    pct_removed = (cancellation_count / current_rows * 100) if current_rows > 0 else 0
    print(f"  ✓ Removed {cancellation_count:,} cancellations ({pct_removed:.2f}%)")
    current_rows = new_rows
    
    # ── STEP 2: Handle Missing CustomerID ───────────────────
    # Business Rationale: CustomerID is missing for guest checkouts. We assign
    # a dummy ID of 0 so that we can retain the sales volume from guest checkouts
    # without dropping them, while still distinguishing them from registered users.
    print("Step 2: Handling Missing CustomerID...")
    missing_cust_mask = df['CustomerID'].isna()
    missing_cust_count = missing_cust_mask.sum()
    df['CustomerID'] = df['CustomerID'].fillna(0).astype(int)
    print(f"  ✓ Handled {missing_cust_count:,} missing CustomerIDs (guest customers)")
    
    # ── STEP 3: Fix Negative Quantities ───────────────────────
    # Business Rationale: Negative quantities indicate returns or administrative
    # adjustments. Since we've already removed cancellations, remaining negative
    # quantities are likely anomalies or errors that would skew average order size.
    print("Step 3: Fixing Negative Quantities...")
    neg_qty_mask = df['Quantity'] < 0
    neg_qty_count = neg_qty_mask.sum()
    df = df[~neg_qty_mask]
    new_rows = len(df)
    print(f"  ✓ Removed {neg_qty_count:,} negative quantities")
    current_rows = new_rows
    
    # ── STEP 4: Fix Negative Prices ─────────────────────────
    # Business Rationale: Negative unit prices are usually bad debt adjustments,
    # vouchers, or data entry errors, which do not represent standard retail sales.
    print("Step 4: Fixing Negative Unit Prices...")
    neg_price_mask = df['UnitPrice'] < 0
    neg_price_count = neg_price_mask.sum()
    df = df[~neg_price_mask]
    new_rows = len(df)
    print(f"  ✓ Removed {neg_price_count:,} negative prices")
    current_rows = new_rows
    
    # ── STEP 5: Calculate Total Price ─────────────────────────
    # Business Rationale: TotalPrice (Quantity * UnitPrice) is the primary gross
    # revenue metric for each line item, crucial for customer lifetime value (CLV)
    # and cohort revenue analysis.
    print("Step 5: Calculating Total Price...")
    df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
    print("  ✓ TotalPrice column created")
    
    # ── STEP 6: Convert InvoiceDate to Datetime ─────────────
    # Business Rationale: Parsing date strings into datetime objects enables
    # time-series analysis, cohort grouping, and temporal feature extraction.
    print("Step 6: Converting InvoiceDate to Datetime...")
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='raise')
    print("  ✓ Date converted to datetime")
    
    # ── STEP 7: Extract Time Features ─────────────────────────
    # Business Rationale: Extracting hour, day, month, and year allows us to
    # analyze sales trends by time of day, day of the week, and seasonal patterns.
    print("Step 7: Extracting Time Features...")
    df['Year'] = df['InvoiceDate'].dt.year
    df['Month'] = df['InvoiceDate'].dt.month
    df['Day'] = df['InvoiceDate'].dt.day
    df['Hour'] = df['InvoiceDate'].dt.hour
    df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
    print("  ✓ Time features extracted")
    
    # ── STEP 8: Remove Outliers (UnitPrice > 1000) ───────────
    # Business Rationale: Extremely high prices (like manual entries, postage, or
    # wholesale outliers) distort averages and segmentations; removing them keeps
    # the dataset focused on standard consumer behavior.
    print("Step 8: Removing Outliers (UnitPrice > 1000)...")
    outlier_mask = df['UnitPrice'] > 1000
    outlier_count = outlier_mask.sum()
    df = df[~outlier_mask]
    new_rows = len(df)
    print(f"  ✓ Removed {outlier_count:,} wholesale outliers")
    current_rows = new_rows
    
    # ── STEP 9: Remove Duplicates ───────────────────────────
    # Business Rationale: Duplicate rows represent redundant transmissions or
    # system glitches that artificially inflate transaction volume and revenue.
    print("Step 9: Removing Duplicates...")
    duplicate_cols = ['InvoiceNo', 'StockCode', 'CustomerID', 'InvoiceDate']
    duplicate_count = df.duplicated(subset=duplicate_cols).sum()
    df = df.drop_duplicates(subset=duplicate_cols)
    new_rows = len(df)
    print(f"  ✓ Removed {duplicate_count:,} duplicates")
    current_rows = new_rows
    
    # ── STEP 10: Standardize Country Names ──────────────────
    # Business Rationale: Extra whitespaces in country names create separate,
    # redundant country keys. Standardizing them ensures accurate geographical grouping.
    print("Step 10: Standardizing Country Names...")
    df['Country'] = df['Country'].astype(str).str.strip()
    print("  ✓ Country names standardized")
    
    # ── SAVE AND SUMMARIZE ───────────────────────────────────
    print("\nSaving Cleaned Dataset...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
    print(f"  ✓ Saved to: {OUTPUT_PATH}")
    
    final_rows = len(df)
    rows_removed = original_rows - final_rows
    removal_pct = (rows_removed / original_rows * 100) if original_rows > 0 else 0
    
    print("\n" + "=" * 60)
    print("DATA CLEANING SUMMARY")
    print("=" * 60)
    print(f"  Original rows      : {original_rows:,}")
    print(f"  Final rows         : {final_rows:,}")
    print(f"  Rows removed       : {rows_removed:,}")
    print(f"  Removal percentage : {removal_pct:.2f}%")
    print("=" * 60)

except Exception as e:
    print(f"\n[ERROR] An error occurred during data cleaning: {e}")
    raise

NOTEBOOK 02: DATA CLEANING & PREPROCESSING

[INFO] Loading raw backup from: data\raw\OnlineRetail_raw.csv


[OK]   Loaded 541,909 rows successfully.

Step 1: Removing Cancellations...
  ✓ Removed 9,288 cancellations (1.71%)
Step 2: Handling Missing CustomerID...
  ✓ Handled 134,697 missing CustomerIDs (guest customers)
Step 3: Fixing Negative Quantities...


  ✓ Removed 1,336 negative quantities
Step 4: Fixing Negative Unit Prices...
  ✓ Removed 2 negative prices
Step 5: Calculating Total Price...
  ✓ TotalPrice column created
Step 6: Converting InvoiceDate to Datetime...
  ✓ Date converted to datetime
Step 7: Extracting Time Features...


  ✓ Time features extracted
Step 8: Removing Outliers (UnitPrice > 1000)...
  ✓ Removed 54 wholesale outliers
Step 9: Removing Duplicates...


  ✓ Removed 10,524 duplicates
Step 10: Standardizing Country Names...
  ✓ Country names standardized

Saving Cleaned Dataset...


  ✓ Saved to: data\cleaned\OnlineRetail_cleaned.csv

DATA CLEANING SUMMARY
  Original rows      : 541,909
  Final rows         : 520,705
  Rows removed       : 21,204
  Removal percentage : 3.91%
